In [0]:
dbutils.widgets.text("ConfigId", "11")
ConfigId = dbutils.widgets.get("ConfigId")

dbutils.widgets.text("pipeLineId", "1")
pipeLineId = dbutils.widgets.get("pipeLineId")

dbutils.widgets.text("SourceTypeId", "6")
SourceTypeId = dbutils.widgets.get("SourceTypeId")

dbutils.widgets.text("LastFullLoadExecutionTime", "2023-11-07T09:01:38.581210Z")
LastFullLoadExecutionTime = dbutils.widgets.get("LastFullLoadExecutionTime")

dbutils.widgets.text("ScheduleInSeconds", "3600")
ScheduleInSeconds = dbutils.widgets.get("ScheduleInSeconds")

job_id=dbutils.widgets.text("job_id","-1")
job_id=dbutils.widgets.get("job_id")

run_id=dbutils.widgets.text("run_id","-1")
run_id=dbutils.widgets.get("run_id")

dbutils.widgets.text('EmailNotificationID','3')
EmailNotificationID = dbutils.widgets.get('EmailNotificationID')

dbutils.widgets.text('Env','Dev')
Env = dbutils.widgets.get('Env')

dbutils.widgets.text('ExternalLocation_raw',"/mntprod_raw")
ExternalLocation_raw = dbutils.widgets.get('ExternalLocation_raw')
deltaTablePath_iothub=ExternalLocation_raw+'/iothub/FACT_DeviceTwinEvents/'


dbutils.widgets.text("ExternalLocationName", "mntprod")
ExternalLocationName = dbutils.widgets.get("ExternalLocationName")
filePathPrefix = '/dbfs/'+ExternalLocationName

dbutils.widgets.text("CatalogName", "cd_prod")
CatalogName = dbutils.widgets.get("CatalogName")

In [0]:
spark.conf.set("spark.databricks.delta.formatCheck.enabled", "false")

In [0]:
%run ../Configurations/Init_Scripts

In [0]:
%run ../Configurations/EmailNotificationConfiguration

In [0]:

import json
from azure.iot.hub import IoTHubRegistryManager
from azure.iot.hub.protocol.models import QuerySpecification
from datetime import datetime

import traceback
pipelinename = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
pipelinename = pipelinename.rsplit('/', 1)[-1]
display(pipelinename)

In [0]:
IOTHubEventStreamConnectionString = dbutils.secrets.get(scope="ABV_AKV_ADB_SCOPE", key="IOTHubEventStreamConnectionString")

IOTHubConnectionString = dbutils.secrets.get(scope="ABV_AKV_ADB_SCOPE", key="IoTHubConnectionString")

ehConf = {}
ehConf['eventhubs.connectionString'] = sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(IOTHubEventStreamConnectionString)

# deltaTablePath_iothub = '/mnt/raw/iothub/FACT_DeviceTwinEvents/'
deltaTablePath_iothub=
checkPointLocation = "/_checkpoints/"

# processedFileList = []
# test = []
fileDetails = dict()

# filePathPrefix = '/dbfs/mnt'
DestinationContainerPath = 'raw'
DestinationFolderPath = 'DeviceTwin'
DestinationFileName = 'DeviceTwin_<TimeStamp>.json'



In [0]:
lastFullLoadExecutionTime_Time = datetime.strptime(LastFullLoadExecutionTime, "%Y-%m-%dT%H:%M:%S.%fZ")
scheduleInSeconds_int = int(ScheduleInSeconds)

In [0]:
def processFullLoad():
    try:
        deviceTwinArray = []
        twininfo = []
        # print('Device twin data load started')
        currentTime = datetime.now().strftime("%Y%m%d%H%M%S")
        DestinationFileName_tmstamp = DestinationFileName.replace('<TimeStamp>',currentTime)
        # print(DestinationFolderPath)

        destinationPath = filePathPrefix+'/'+DestinationContainerPath+'/'+DestinationFolderPath+'/'+DestinationFileName_tmstamp

        registry_manager = IoTHubRegistryManager(IOTHubConnectionString)
        query = QuerySpecification(query= "SELECT * FROM devices")
        
        results = registry_manager.query_iot_hub(query)
        twininfo = results.items
        continuationToken = results.continuation_token
        while(continuationToken is not None):
            results = registry_manager.query_iot_hub(query,continuation_token=continuationToken)
            twininfo = twininfo+results.items
            continuationToken = results.continuation_token
        # print('Device twin data load end')
        # print('Device twin data read from registry and load into json array started')
        for twin in twininfo:
            devicetwin = dict()
            twindata = twin.as_dict()
            device_id = twindata.get('device_id')
            productType = twindata.get('properties').get('reported').get('productType')
            if(productType is None):
                if(device_id.startswith( 'CT' )):
                    productType = 'CT'
                elif(device_id.startswith( 'RS' )):
                    productType = 'RS'
                else:
                    productType = 'COM3'

            devicetwin['twinData'] = json.dumps(twindata)
            devicetwin['device_id'] = device_id
            devicetwin['ProductType'] = productType
            deviceTwinArray.append(devicetwin)

        # print('Device twin data read from registry and load into json array started')
        # print('Load data to json file started')

        # with open(destinationPath, "w") as outfile:
        #     outfile.write(json.dumps(deviceTwinArray))
        # print('Load data to json file end')

        fileDetails['PipelineStatus'] = 'Succeeded'
        # processedFileList.append(fileDetails)
        # logIntoIngestionLogTable(processedFileList,'ADB_LoadDeviceTwin')
        return deviceTwinArray
    except Exception as e:
        fileDetails['PipelineStatus'] = 'Failed'
        fileDetails['ErrorMessage'] = str(e)
        # processedFileList.append(fileDetails)
        # logIntoIngestionLogTable(processedFileList,'ADB_LoadDeviceTwin')    
        print(str(e))
        # raise

In [0]:
def upsertToDelta(microBatchOutputDF, batchId):      
    batchEnd(q,batchId)
    print("Running for BatchID: {0}".format(batchId))    
    global lastFullLoadExecutionTime_Time

    Log = []
    LogDict = dict()
    LogDict['FileNameUUID'] = str(uuid.uuid4())
    LogDict['ConfigId'] = ConfigId
    LogDict['FileNameDeviceTypeCd'] = 'DeviceTwinStream'
    LogDict['ExternalSerialNbr'] = 'DeviceTwinStream'
    LogDict['InternalSerialNbr'] = 'DeviceTwinStream'
    LogDict['FileNameMessageTypeCd'] = 'DeviceTwinStream'
    LogDict['FileNameDtTmstmp'] = ''
    LogDict['FileNameApplicatorPortCd'] = ''
    LogDict['FileNameCycleNbr'] = ''
    LogDict['SourceFilePath'] = ''
    LogDict['SourceFileName'] = ''
    LogDict['SourceFileSize'] = 0
    LogDict['SourceTypeId'] = SourceTypeId
    LogDict['DeviceType'] = 'DeviceTwin-IOTHub'
    LogDict['LogType'] = 'Ingestion'
    LogDict['DeviceId'] = ''
    LogDict['RunID'] = batchId
    LogDict['RecdCnt'] = microBatchOutputDF.count()
    Log.append(LogDict)

    df_log = spark.createDataFrame(Log)
    df_log = (df_log.withColumn("logstartdate",current_timestamp())
                .withColumn("logEnddate",current_timestamp())
                .withColumn("RawFileModificationTime",current_timestamp())                
                .select('FileNameDeviceTypeCd','ExternalSerialNbr','InternalSerialNbr','FileNameMessageTypeCd'
                        ,'FileNameDtTmstmp','FileNameApplicatorPortCd','FileNameCycleNbr','ConfigId','SourceFilePath'
                        ,'SourceFileName','SourceFileSize', 'SourceTypeId','DeviceType','LogType','DeviceId'
                        ,"RecdCnt",'FileNameUUID',"logstartdate","logEnddate","RawFileModificationTime","RunID"))
    
    loadlogProcessesDeltaTable(df_log,deltaTablePath_iothub,'ADB_IOTStreamProcessing','InProgress','')
    loadAuditTables_Ingestion_Log(df_log,deltaTablePath_iothub,'ADB_IOTStreamProcessing','InProgress','')
    
    try:
        log_df = df_log.select(col('ConfigId').alias('ConfigID'), col('SourceTypeId').alias('SourceTypeID'), col('SourceFilePath').alias('Source')) \
                                         .withColumn('Destination', lit(str(deltaTablePath_iothub))) \
                                         .withColumn('Run_ID', lit(str(batchId))) \
                                         .withColumn('Job_ID', lit(str(job_id)))
        
        Microbatch_df = df_log

        # raise Exception("No Exception: Manual Failure")

        # deviceTwinArrayJson
        microBatchOutputDF = (microBatchOutputDF.withColumn('CreatedBy',lit("ADB_IOTStreamProcessing"))
                                                .withColumn('CreatedDt',lit(current_timestamp()))
                                                .withColumn('UpdatedBy',lit("ADB_IOTStreamProcessing"))
                                                .withColumn('UpdatedDt',lit(current_timestamp())))

        microBatchOutputDF.write.format('delta').mode("append").save(deltaTablePath_iothub)

        # Run Full Load based on Schedule
        currentTime = datetime.now()
        timeDiffSecs = (currentTime - lastFullLoadExecutionTime_Time).total_seconds()
        # print(lastFullLoadExecutionTime_Time)

        if(timeDiffSecs >= scheduleInSeconds_int):

            lastFullLoadExecutionTime_Time = datetime.now()
            print('FullLoad')
            deviceTwinArray = processFullLoad()
            df_FullLoad = (spark.createDataFrame(deviceTwinArray)
                    .withColumn("operationTimestamp", lit(current_timestamp()).cast("string"))
                    .withColumn("operationType", lit('FullLoad').cast("string"))
                    .withColumn("iothub-message-schema", lit('FullLoad').cast("string"))
                    .select('device_id','productType','twinData','operationType','iothub-message-schema','operationTimestamp')
                    .withColumn('CreatedBy',lit("ADB_IOTStreamProcessing"))
                    .withColumn('CreatedDt',lit(current_timestamp()))
                    .withColumn('UpdatedBy',lit("ADB_IOTStreamProcessing"))
                    .withColumn('UpdatedDt',lit(current_timestamp())))
            
            df_FullLoad.write.format('delta').mode("append").save(deltaTablePath_iothub)

        # print("---------------------------------------------------------")
        spark.sql("clear cache")
        # batchEnd(q,stop_process)
        loadAuditTables_Ingestion_Log(df_log,deltaTablePath_iothub,'ADB_IOTStreamProcessing','Succeeded','')
        loadlogProcessesDeltaTable(df_log,deltaTablePath_iothub,'ADB_IOTStreamProcessing','Succeeded','')
        
    except Exception as exp:
        ExceptionTraceback = traceback.format_exc()
        ErrorMessage = ExceptionTraceback + str(exp)

        loadlogProcessesDeltaTable(df_log,deltaTablePath_iothub,'ADB_IOTStreamProcessing','Failed',str(exp))
        loadAuditTables_Ingestion_Log(df_log,deltaTablePath_iothub,'ADB_IOTStreamProcessing','Failed',str(exp))
        
        logIntoStreamLogTable(log_df,"ADB_IOTStreamProcessing","Failed",Microbatch_df,ErrorMessage)
        streamLogEmailNotification(EmailNotificationID,Microbatch_df, pipelinename, Env)
        print(ExceptionTraceback)
        # raise 

In [0]:
df = spark.readStream.format("eventhubs").options(**ehConf).load()

readInStreamBody = (df
                    .withColumn("twinData", col('body').cast("string"))
                    .withColumn("productType", lit(None).cast("string"))
                    .withColumn("device_id", col('properties.deviceId').cast("string"))
                    .withColumn("operationTimestamp", col('properties.operationTimestamp'))
                    .withColumn("operationType", col('properties.opType').cast("string"))
                    .withColumn("iothub-message-schema", col('properties.iothub-message-schema').cast("string"))
                    .select('device_id','productType','twinData','operationType','iothub-message-schema','operationTimestamp')
                    )
# display(readInStreamBody)


In [0]:
q=(readInStreamBody.writeStream
                  .format("delta")
                  .queryName("V2_Ingestion_iot_Stream")
                  .trigger(processingTime='10 seconds')
                  .foreachBatch(upsertToDelta)
                  .option("checkpointLocation", deltaTablePath_iothub+checkPointLocation)
                  .outputMode("update")
                  .start()
)


In [0]:
stop_process = 1
stop_process =  graceStop(q,1)

In [0]:
q.awaitTermination()